# Lab 17-05 · Matched Filters and Decisions

Five parts, each built on `detection.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import detection as dd
rng = np.random.default_rng(1705)

## Part 1 · The matched filter's SNR

*Predict:* $2E/N_0$ for a 1 µs, 1 V rectangular pulse with $N_0 = 2 \times 10^{-8}$ V²/Hz, and for a half-sine and a chirp of the same energy.

In [2]:
T, dt, n0 = 1e-6, 1e-8, 2e-8
t = np.arange(int(T / dt)) * dt
rect = np.ones(len(t))
E = dd.energy(rect, dt)
pulses = {"rectangle": rect,
          "half-sine": np.sin(np.pi * t / T),
          "chirp": np.cos(2 * np.pi * (5e6 * t + 0.5 * 20e6 / T * t ** 2))}
for name, p in pulses.items():
    p = p * math.sqrt(E / dd.energy(p, dt))
    snr, th = dd.mf_peak_snr_mc(p, n0, dt, 20000, rng)
    print(f"{name:10s}: measured {snr:6.1f}, predicted 2E/N0 = {th:.1f}")
    assert abs(snr / th - 1) < 0.04

rectangle : measured  101.1, predicted 2E/N0 = 100.0
half-sine : measured  100.0, predicted 2E/N0 = 100.0
chirp     : measured   99.2, predicted 2E/N0 = 100.0


## Part 2 · Mismatch

*Predict:* where the one-pole filter's SNR peaks, and how much it loses there.

In [3]:
x = np.linspace(0.2, 5, 200)
ratio = [dd.one_pole_ratio(v) for v in x]
best = optimize.minimize_scalar(lambda v: -dd.one_pole_ratio(v), bounds=(0.1, 5), method="bounded")
print(f"optimum T/tau = {best.x:.4f}, ratio {dd.one_pole_ratio(best.x):.4f} ({10 * math.log10(dd.one_pole_ratio(best.x)):.2f} dB)")
plt.plot(x, 10 * np.log10(ratio)); plt.xlabel("T / tau"); plt.ylabel("SNR relative to matched filter (dB)"); plt.show()
assert abs(best.x - 1.2564) < 1e-3

optimum T/tau = 1.2564, ratio 0.8145 (-0.89 dB)


## Part 3 · Signal space

*Predict:* the dimension of Example 17-05.3's signal set, and of four random waveforms of three samples each.

In [4]:
n = 1000; ddt = 1 / n
s1 = np.ones(n); s2 = np.r_[np.ones(n // 2), -np.ones(n // 2)]; s3 = np.r_[np.ones(n // 2), np.zeros(n // 2)]
B = dd.gram_schmidt([s1, s2, s3], ddt)
print("Example 17-05.3: dimension", len(B), " coordinates", np.round(dd.coordinates([s1, s2, s3], B, ddt), 4).tolist())
R = rng.standard_normal((4, 3))
print("four random 3-sample waveforms: dimension", len(dd.gram_schmidt(list(R), 1.0)))
assert len(B) == 2

Example 17-05.3: dimension 2  coordinates [[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]]
four random 3-sample waveforms: dimension 3


## Part 4 · Error rates

*Predict:* the BER of antipodal, orthogonal and on-off signalling at 0–10 dB, and PAM4's symbol-error rate against its formula and nearest-neighbour approximation.

In [5]:
sets = {"antipodal": np.array([[1.0], [-1.0]]), "orthogonal": np.array([[1.0, 0.0], [0.0, 1.0]]), "on-off": np.array([[0.0], [math.sqrt(2.0)]])}
grid = np.arange(0, 11, 2)
for name, pts in sets.items():
    eb = np.mean(np.sum(pts ** 2, axis=1))     # one bit per symbol
    sim = [dd.simulate_ser(pts, eb / 10 ** (g / 10), 300_000, rng) for g in grid]
    th = [dd.ber_bpsk(10 ** (g / 10)) if name == "antipodal" else dd.ber_orthogonal(10 ** (g / 10)) for g in grid]
    print(name, [f"{a:.2e}/{b:.2e}" for a, b in zip(sim, th)])
    plt.plot(grid, sim, "o", label=f"{name} (sim)"); plt.plot(grid, th, "-")
plt.yscale("log"); plt.xlabel("Eb/N0 (dB)"); plt.ylabel("BER"); plt.legend(); plt.show()
pam4 = np.array([[-3.0], [-1.0], [1.0], [3.0]])
for esdb in (10, 14):
    n0 = np.mean(pam4 ** 2) / 10 ** (esdb / 10)
    print(f"PAM4 at Es/N0 = {esdb} dB: simulated {dd.simulate_ser(pam4, n0, 1_000_000, rng):.3e}, formula {dd.ser_pam(4, 10 ** (esdb / 10)):.3e}, nearest-neighbour {dd.nearest_neighbour(pam4, n0):.3e}")

antipodal ['7.95e-02/7.86e-02', '3.72e-02/3.75e-02', '1.22e-02/1.25e-02', '2.33e-03/2.39e-03', '1.90e-04/1.91e-04', '3.33e-06/3.87e-06']


orthogonal ['1.59e-01/1.59e-01', '1.04e-01/1.04e-01', '5.62e-02/5.65e-02', '2.27e-02/2.30e-02', '6.01e-03/6.00e-03', '7.90e-04/7.83e-04']
on-off ['1.59e-01/1.59e-01', '1.04e-01/1.04e-01', '5.71e-02/5.65e-02', '2.31e-02/2.30e-02', '5.98e-03/6.00e-03', '7.90e-04/7.83e-04']
PAM4 at Es/N0 = 10 dB: simulated 3.406e-02, formula 3.413e-02, nearest-neighbour 3.413e-02


PAM4 at Es/N0 = 14 dB: simulated 1.126e-03, formula 1.144e-03, nearest-neighbour 1.144e-03


## Part 5 · Pulse compression

*Predict:* the compression ratio and main-lobe width for a 10 µs chirp sweeping 10 MHz, and whether an echo 6 dB below the noise per sample can be found.

In [6]:
fs = 50e6
ref = dd.chirp(10e-6, 10e6, fs)
print(f"time-bandwidth product {10e-6 * 10e6:.0f}; -4 dB main-lobe width {dd.mainlobe_width(dd.compress(ref, ref), fs, 2 / math.pi) * 1e9:.0f} ns (1/B = 100 ns)")
rx = np.zeros(3000, dtype=complex)
delay = 1234
rx[delay: delay + len(ref)] = 0.5 * ref
rx += (rng.standard_normal(len(rx)) + 1j * rng.standard_normal(len(rx))) * math.sqrt(0.5)
y = np.abs(dd.compress(rx, ref))
found = int(np.argmax(y)) - (len(ref) - 1)
print(f"echo placed at sample {delay}, found at {found}")
plt.plot(y); plt.xlabel("sample"); plt.ylabel("|compressed|"); plt.show()
assert found == delay

time-bandwidth product 100; -4 dB main-lobe width 100 ns (1/B = 100 ns)
echo placed at sample 1234, found at 1234
